# 🚁 NOTEBOOK 08: TEST SET PROCESSING (BLIND TEST)
**Project:** Intelligent Drone Flight Maneuver Recognition  
**Phase:** Testing Data Preparation  

## 🎯 Mục tiêu
1. **Input:** File zip dữ liệu Test (VD: `Final_Exam.zip`) trong folder `data/raw/test`.
2. **Segmentation:** Cắt dữ liệu dựa trên file Annotation (chỉ lấy đoạn có bấm nút).
3. **Feature Extraction:** Áp dụng bộ công thức 30 Features (Top selected) hoặc Full Features.
4. **Output:**
   - `X_test.csv`: Dữ liệu đặc trưng (Input cho Model).
   - `y_test.csv`: Nhãn thực tế (Ground Truth - Dùng để chấm điểm).

In [26]:
from google.colab import drive
import pandas as pd
import numpy as np
import os
import shutil
import zipfile
from pathlib import Path
from tqdm import tqdm
from scipy.stats import kurtosis, skew

drive.mount('/content/drive')

# --- CẤU HÌNH ---
BASE_DIR = Path('/content/drive/MyDrive/Drone_Project_2025')
RAW_TEST_DIR = BASE_DIR / 'data/raw/test'
PROCESSED_DIR = BASE_DIR / 'data/processed'
FEATURES_DIR = BASE_DIR / 'data/features'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

# Cấu hình Window (Phải KHỚP 100% với lúc Train)
SAMPLING_RATE = 100
WINDOW_SIZE = 100 # 1.0s
STEP_SIZE = 20    # 0.2s

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 🛠️ 1. HÀM XỬ LÝ RAW (UNZIP & SYNC)

In [27]:
def process_raw_test_file(zip_path):
    temp_dir = Path('/content/temp_extract')
    if temp_dir.exists(): shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True, exist_ok=True)

    # 1. Giải nén
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(temp_dir)

    # 2. Đọc CSV
    files = {
        'acc': temp_dir / 'Accelerometer.csv',
        'gyro': temp_dir / 'Gyroscope.csv',
        'anno': temp_dir / 'Annotation.csv'
    }
    if not all(f.exists() for f in files.values()): return None

    # Load & Sync
    df_acc = pd.read_csv(files['acc'])
    df_acc['time'] = pd.to_datetime(df_acc['time'], unit='ns')
    df_acc = df_acc.rename(columns={'x': 'acc_x', 'y': 'acc_y', 'z': 'acc_z'})
    df_acc = df_acc[['time', 'acc_x', 'acc_y', 'acc_z']].sort_values('time')

    df_gyro = pd.read_csv(files['gyro'])
    df_gyro['time'] = pd.to_datetime(df_gyro['time'], unit='ns')
    df_gyro = df_gyro.rename(columns={'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})
    df_gyro = df_gyro[['time', 'gyro_x', 'gyro_y', 'gyro_z']].sort_values('time')

    df_merged = pd.merge_asof(df_acc, df_gyro, on='time', direction='nearest', tolerance=pd.Timedelta('20ms'))
    df_merged = df_merged.dropna()

    # 3. Cắt segment dựa trên Annotation (Chỉ lấy đoạn bấm nút)
    df_anno = pd.read_csv(files['anno'])
    df_anno['end_time'] = pd.to_datetime(df_anno['time'], unit='ns')
    df_anno['start_time'] = df_anno['end_time'] - pd.to_timedelta(df_anno['millisecond_press_duration'], unit='ms')
    df_anno = df_anno.sort_values('start_time')

    # MAP NHÃN NẾU CẦN (Cho kịch bản Maneuver)
    is_scenario = 'exam' in zip_path.name.lower() or 'test' in zip_path.name.lower()
    maneuver_order = ['MOVE_LEFT', 'MOVE_RIGHT', 'MOVE_FORWARD', 'MOVE_BACKWARD']
    maneuver_idx = 0

    segments = []

    print(f"\n📂 Đang xử lý file: {zip_path.name}")
    for i, row in df_anno.iterrows():
        label = row['text']

        # Logic Map Maneuver (nếu trong kịch bản)
        if is_scenario and label == 'Maneuver':
            if maneuver_idx < len(maneuver_order):
                label = maneuver_order[maneuver_idx]
                maneuver_idx += 1

        # Cắt data
        mask = (df_merged['time'] >= row['start_time']) & (df_merged['time'] <= row['end_time'])
        seg = df_merged.loc[mask].copy()

        if not seg.empty:
            seg['label'] = str(label).upper()
            seg['segment_id'] = i
            segments.append(seg)
            print(f"  -> Segment {i}: {label} ({len(seg)} samples)")

    if segments:
        return pd.concat(segments, ignore_index=True)
    return None

## ✂️ 2. HÀM WINDOWING & FEATURE EXTRACTION
Áp dụng công thức tính ZCR, Jerk, Kurtosis...

In [28]:
# ==============================================================================
# CẬP NHẬT HÀM FEATURE EXTRACTION (KHỚP 100% VỚI TRAINING)
# ==============================================================================
from scipy.stats import kurtosis, skew

def extract_features(window_data):
    features = {}
    axis_names = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

    # Giả định sampling rate là 100Hz (như cấu hình chung)
    fs = 100.0

    for i, axis in enumerate(axis_names):
        signal = window_data[:, i]

        # --- 1. TIME DOMAIN (Cơ bản) ---
        features[f'{axis}_mean'] = np.mean(signal)
        features[f'{axis}_std'] = np.std(signal)
        features[f'{axis}_max'] = np.max(signal)
        features[f'{axis}_min'] = np.min(signal)
        features[f'{axis}_range'] = np.max(signal) - np.min(signal)
        features[f'{axis}_rms'] = np.sqrt(np.mean(signal**2))

        # --- 2. PHYSICS (Hình dạng & Va chạm) ---
        features[f'{axis}_kurtosis'] = kurtosis(signal)
        features[f'{axis}_skew'] = skew(signal)

        # --- 3. DYNAMICS (ZCR & Jerk) ---
        centered = signal - np.mean(signal)
        zcr = ((centered[:-1] * centered[1:]) < 0).sum()
        features[f'{axis}_zcr'] = zcr

        jerk = np.diff(signal)
        features[f'{axis}_jerk_mean'] = np.mean(np.abs(jerk))
        features[f'{axis}_jerk_max'] = np.max(np.abs(jerk))

        # --- 4. FREQUENCY DOMAIN (FFT & Dom Freq) ---
        # Đây là phần Notebook 08 cũ bị thiếu gây ra lỗi KeyError
        fft_vals = np.abs(np.fft.rfft(signal))
        freqs = np.fft.rfftfreq(len(signal), d=1/fs)

        # Bỏ thành phần DC (0Hz)
        if len(fft_vals) > 1:
            fft_vals = fft_vals[1:]
            freqs = freqs[1:]

            if len(fft_vals) > 0:
                features[f'{axis}_energy'] = np.sum(fft_vals**2) / len(fft_vals)
                features[f'{axis}_fft_max'] = np.max(fft_vals)

                # Tìm tần số trội (Dominant Frequency)
                dom_idx = np.argmax(fft_vals)
                features[f'{axis}_dom_freq'] = freqs[dom_idx]
            else:
                features[f'{axis}_energy'] = 0
                features[f'{axis}_fft_max'] = 0
                features[f'{axis}_dom_freq'] = 0
        else:
            features[f'{axis}_energy'] = 0
            features[f'{axis}_fft_max'] = 0
            features[f'{axis}_dom_freq'] = 0

    return features

In [29]:
def process_test_pipeline():
    zip_files = list(RAW_TEST_DIR.glob('*.zip'))
    if not zip_files:
        print("⚠️ Không tìm thấy file zip nào trong data/raw/test/")
        return

    all_X = []
    all_y = []

    for zip_file in zip_files:
        df_clean = process_raw_test_file(zip_file)
        if df_clean is None: continue

        for seg_id in df_clean['segment_id'].unique():
            seg_df = df_clean[df_clean['segment_id'] == seg_id]
            label = seg_df['label'].iloc[0]

            feature_cols = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']
            for i in range(0, len(seg_df) - WINDOW_SIZE + 1, STEP_SIZE):
                window = seg_df.iloc[i:i+WINDOW_SIZE][feature_cols].values
                feats = extract_features(window)
                all_X.append(feats)
                all_y.append(label)

    if all_X:
        X_df = pd.DataFrame(all_X)
        y_df = pd.Series(all_y, name='label')

        X_df.to_csv(FEATURES_DIR / 'X_test.csv', index=False)
        y_df.to_csv(FEATURES_DIR / 'y_test.csv', index=False)

        print(f"\n✅ HOÀN TẤT! Đã tạo X_test.csv ({X_df.shape}) và y_test.csv")
    else:
        print("⚠️ Không tạo được dữ liệu nào.")

In [30]:
# CHẠY PIPELINE
process_test_pipeline()


📂 Đang xử lý file: Test-2026-01-06_03-45-25.zip
  -> Segment 0: Idle (307 samples)
  -> Segment 1: Takeoff (155 samples)
  -> Segment 2: Hover (247 samples)
  -> Segment 4: Hover (205 samples)
  -> Segment 5: MOVE_LEFT (164 samples)
  -> Segment 6: MOVE_RIGHT (164 samples)
  -> Segment 7: Hover (170 samples)
  -> Segment 8: MOVE_FORWARD (155 samples)
  -> Segment 9: MOVE_BACKWARD (126 samples)
  -> Segment 10: Hover (196 samples)
  -> Segment 11: Takeoff (150 samples)
  -> Segment 12: Idle (208 samples)

✅ HOÀN TẤT! Đã tạo X_test.csv ((59, 84)) và y_test.csv
